# Wi-Fi CSI kalite-farkındalıklı HAR — Kaggle optimizasyonu

Bu notebook, gated CNN–LSTM modelini GPU üzerinde Optuna ile tarar ve en iyi ayarla tam eğitim yapar.

Kaggle'da sağdaki **Add Input** düğmesiyle şu iki özel veri kümesini ekleyin:
1. `wifi-csi-quality-code` — teslim paketindeki kod klasörü/ZIP'i.
2. `wifi-csi-preprocessed-cache` — yerel `03_On_Isleme_Cache` klasörünün TAR arşivi.

Notebook ayarlarından **Accelerator → GPU** seçin. İlk çalıştırmada Internet'i açın; Optuna kurulunca kapatabilirsiniz.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, tarfile, zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/wifi_csi_search')
WORK.mkdir(parents=True, exist_ok=True)
print('Bağlı girdiler:')
for path in sorted(INPUT.iterdir()):
    print(' -', path)


In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU görünmüyor. Notebook Settings > Accelerator > GPU seçin.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)


In [ ]:
# Kod paketini bul veya aç.
code_candidates = list(INPUT.rglob('pyproject.toml'))
if code_candidates:
    CODE = code_candidates[0].parent
else:
    archives = [p for p in INPUT.rglob('*') if p.is_file() and 'wifi_csi_quality_code' in p.name.lower()]
    assert archives, 'Kod girdisi bulunamadı.'
    CODE = WORK / 'code'
    CODE.mkdir(exist_ok=True)
    shutil.unpack_archive(str(archives[0]), str(CODE))
    projects = list(CODE.rglob('pyproject.toml'))
    assert projects, 'Arşiv içinde pyproject.toml bulunamadı.'
    CODE = projects[0].parent

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna>=4,<5', 'pandas>=2'], check=True)
os.environ['PYTHONPATH'] = str(CODE/'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, str(CODE/'src'))
import wifi_csi_quality
print('Kod kökü:', CODE)


In [ ]:
# Ön işleme önbelleğini bul; TAR/ZIP olarak yüklendiyse bir kez aç.
train_indexes = list(INPUT.rglob('csi_bench_train_id_cache_index.jsonl'))
if not train_indexes:
    cache_archives = [p for p in INPUT.rglob('*') if p.is_file() and 'cache' in p.name.lower() and (p.suffix in {'.zip', '.tar', '.bin'} or p.name.endswith('.tar.gz'))]
    assert cache_archives, 'Ön işleme önbelleği veya arşivi bulunamadı.'
    CACHE_EXTRACT = WORK / 'cache'
    CACHE_EXTRACT.mkdir(exist_ok=True)
    print('Önbellek açılıyor:', cache_archives[0].name)
    if cache_archives[0].suffix == '.bin':
        with tarfile.open(cache_archives[0], 'r:*') as archive:
            archive.extractall(CACHE_EXTRACT, filter='data')
    else:
        shutil.unpack_archive(str(cache_archives[0]), str(CACHE_EXTRACT))
    train_indexes = list(CACHE_EXTRACT.rglob('csi_bench_train_id_cache_index.jsonl'))
assert train_indexes, 'Eğitim indeks dosyası bulunamadı.'
TRAIN_INDEX = train_indexes[0]
CACHE_ROOT = TRAIN_INDEX.parent
VAL_INDEX = CACHE_ROOT / 'csi_bench_val_id_cache_index.jsonl'
assert VAL_INDEX.exists(), 'Doğrulama indeks dosyası bulunamadı.'
first = json.loads(TRAIN_INDEX.read_text(encoding='utf-8').splitlines()[0])
sample_path = CACHE_ROOT / first['cache_relative_path']
assert sample_path.exists(), f'İndeksteki örnek bulunamadı: {sample_path}'
print('Cache kökü:', CACHE_ROOT)
print('Örnek doğrulandı:', sample_path)


## 1. Kısa sistem testi
Bu hücre veri yollarını, kodu ve GPU uyumluluğunu küçük bir örnek üzerinde doğrular.

In [ ]:
smoke_out = WORK / 'smoke_test'
cmd = [sys.executable, '-m', 'wifi_csi_quality.quality_learning',
       '--cache-root', str(CACHE_ROOT), '--train-index', str(TRAIN_INDEX),
       '--val-index', str(VAL_INDEX), '--impairments-config', str(CODE/'configs/impairments_v1.toml'),
       '--training-config', str(CODE/'configs/training_v1.toml'), '--output-dir', str(smoke_out),
       '--model', 'gated_quality_cnn_lstm', '--smoke']
subprocess.run(cmd, check=True)
print('Kısa test başarılı.')


## 2. Optuna taraması
Varsayılan tarama 12 denemedir. İlk çalışma için değiştirmeyin. Başarılı olursa daha sonra 24–30 denemeye çıkarılabilir.

In [ ]:
TRIALS = 12
SEARCH = WORK / 'optuna'
cmd = [sys.executable, '-u', '-m', 'wifi_csi_quality.optuna_search',
       '--cache-root', str(CACHE_ROOT), '--train-index', str(TRAIN_INDEX),
       '--val-index', str(VAL_INDEX), '--impairments-config', str(CODE/'configs/impairments_v1.toml'),
       '--output-dir', str(SEARCH), '--trials', str(TRIALS), '--train-per-class', '2000', '--epochs', '12']
subprocess.run(cmd, check=True)
best = json.loads((SEARCH/'optuna_best.json').read_text(encoding='utf-8'))
best


## 3. En iyi ayarla tam eğitim
Bu hücre yalnızca Optuna taraması tamamlandıktan sonra çalıştırılmalıdır.

In [ ]:
FINAL = WORK / 'final_seed_2029'
cmd = [sys.executable, '-u', '-m', 'wifi_csi_quality.quality_learning',
       '--cache-root', str(CACHE_ROOT), '--train-index', str(TRAIN_INDEX),
       '--val-index', str(VAL_INDEX), '--impairments-config', str(CODE/'configs/impairments_v1.toml'),
       '--training-config', str(SEARCH/'best_training.toml'), '--output-dir', str(FINAL),
       '--model', 'gated_quality_cnn_lstm', '--resume']
subprocess.run(cmd, check=True)
result = json.loads((FINAL/'gated_quality_cnn_lstm_results.json').read_text(encoding='utf-8'))
best_row = max(result['history'], key=lambda row: row['validation']['macro_f1'])
print('En iyi dönem:', result['best_epoch'])
print('Makro-F1:', round(100*best_row['validation']['macro_f1'], 2), '%')
print('Dengeli doğruluk:', round(100*best_row['validation']['balanced_accuracy'], 2), '%')
print('Doğruluk:', round(100*best_row['validation']['accuracy'], 2), '%')
print('ECE:', round(best_row['validation']['ece'], 4))


In [ ]:
# Kaggle Output bölümünden indirilecek tek arşiv oluşturulur.
archive = shutil.make_archive('/kaggle/working/wifi_csi_optuna_results', 'zip', WORK)
print('Hazır:', archive)
